# V3 end to end — Silver V2 → reviewed Silver V3 → Gold V3 → Modeling

This follows the V1/V2 immutable Data Lake pattern. Silver V2 is the pinned V3 Bronze-equivalent input. Review an up-to-300-row News pilot first, freeze the label guide, then create and review all News decisions before publishing Silver V3. Gold V3 creates new leakage-aware splits from the complete Silver V3 pool. The final optional stage trains and compares the two TF-IDF baselines on train/validation only.

V3 uses 27 classes: existing specialist labels plus three provisional News sublabels; sports uses the existing `Sports` label. All machine suggestions require human approval. Publish and modeling flags default to False; complete the review and publishing stages before enabling downstream stages.

To test a pushed feature branch, set `VTC_GIT_REF` in a separate code cell before running `setup`. The setup cell intentionally defaults to `main`.

In [ ]:
from pathlib import Path
import hashlib
import importlib
import json
import os
import subprocess
import sys

REPO_URL = "https://github.com/Ko4rut/vietnamese-multimodal-moderation.git"
BRANCH = os.environ.get("VTC_GIT_REF", "main")
PROJECT_DIR = Path("/content/vmm_v3_project")
if (PROJECT_DIR / ".git").is_dir():
    dirty = subprocess.check_output(["git", "-C", str(PROJECT_DIR), "status", "--porcelain"], text=True)
    if dirty.strip():
        raise RuntimeError("Colab checkout has uncommitted changes; resolve them first")
    subprocess.run(["git", "-C", str(PROJECT_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "checkout", "-B", BRANCH, "FETCH_HEAD"], check=True)
else:
    if PROJECT_DIR.exists() and any(PROJECT_DIR.iterdir()):
        raise RuntimeError("Project directory contains non-Git files")
    subprocess.run(["git", "clone", "--single-branch", "--branch", BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[modeling]"], cwd=PROJECT_DIR, check=True)
SRC_DIR = (PROJECT_DIR / "src").resolve()
if not (SRC_DIR / "vietnamese_topic_classification" / "__init__.py").is_file():
    raise FileNotFoundError(f"Missing project Python sources: {SRC_DIR}")
sys.path[:] = [part for part in sys.path if part != str(SRC_DIR)]
sys.path.insert(0, str(SRC_DIR))
prefix = "vietnamese_topic_classification"
for name in sorted([name for name in sys.modules if name == prefix or name.startswith(prefix + ".")], key=len, reverse=True):
    del sys.modules[name]
importlib.invalidate_caches()

from vietnamese_topic_classification.analysis.news_v3 import build_keyword_report, build_prompt, sample_news_records
from vietnamese_topic_classification.config import get_v3_domain_labels
from vietnamese_topic_classification.data.colab_lake import resolve_mounted_lake
from vietnamese_topic_classification.data.gold import GoldConfig, GoldReader, GoldV3Pipeline
from vietnamese_topic_classification.data.silver import SilverReader, SilverV3Pipeline

print("Source revision:", subprocess.check_output(["git", "-C", str(PROJECT_DIR), "rev-parse", "--short", "HEAD"], text=True).strip())

In [ ]:
from google.colab import drive

MOUNT = "/content/vtsnlp_drive"
drive.mount(MOUNT)
DATA_LAKE_ROOT = resolve_mounted_lake(MOUNT)
print("Mounted Data Lake:", DATA_LAKE_ROOT)

In [ ]:
SILVER_V2_REPORT_NAME = None  # Set an exact filename if multiple reports exist.
reports_dir = DATA_LAKE_ROOT / "04_reports" / "silver" / "v2"
available = sorted(path.name for path in reports_dir.glob("silver_quality_v2_*.json"))
print("Silver V2 reports:", available)
if SILVER_V2_REPORT_NAME is None:
    if len(available) != 1:
        raise ValueError(f"Choose one exact SILVER_V2_REPORT_NAME; found {len(available)}")
    SILVER_V2_REPORT_NAME = available[0]
if Path(SILVER_V2_REPORT_NAME).name != SILVER_V2_REPORT_NAME:
    raise ValueError("Report name must be a filename, not a path")
silver_v2 = SilverReader(DATA_LAKE_ROOT, SILVER_V2_REPORT_NAME)
verified_v2 = SilverV3Pipeline(silver_v2).verify_source()
print("Silver V2 + Silver V1 lineage verified:", verified_v2)

In [ ]:
v3_reports = DATA_LAKE_ROOT / "04_reports" / "silver" / "v3"
source_suffix = f"{verified_v2['sha256'][:12]}_{hashlib.sha256(silver_v2.report_path.read_bytes()).hexdigest()[:8]}"
KEYWORD_REPORT = v3_reports / f"news_keyword_analysis_v3_{source_suffix}.json"
if KEYWORD_REPORT.exists():
    keyword_report = json.loads(KEYWORD_REPORT.read_text(encoding="utf-8"))
else:
    keyword_report = build_keyword_report(silver_v2.iter_records())
    keyword_report["source"] = {"silver_v2_report": SILVER_V2_REPORT_NAME, "silver_v2_sha256": verified_v2["sha256"]}
    KEYWORD_REPORT.parent.mkdir(parents=True, exist_ok=True)
    with KEYWORD_REPORT.open("x", encoding="utf-8", newline="\n") as handle:
        json.dump(keyword_report, handle, ensure_ascii=False, sort_keys=True, indent=2)
        handle.write("\n")
news_profile = keyword_report["classes"].get("News", {})
print("Keyword report:", KEYWORD_REPORT)
print("News rows:", news_profile.get("documents", 0))
print("Top News keywords:", news_profile.get("keywords", [])[:30])
print("News overlap:", keyword_report["news_vs_other_class_keyword_overlap"])
print("Examples:", news_profile.get("examples", []))

In [ ]:
PILOT_SAMPLE_SIZE = 300
TAXONOMY_SUFFIX = hashlib.sha256(json.dumps(get_v3_domain_labels()).encode("utf-8")).hexdigest()[:8]
NEWS_DECISIONS = v3_reports / f"news_review_decisions_v3_{source_suffix}_{TAXONOMY_SUFFIX}.jsonl"
NEWS_PILOT = v3_reports / f"news_pilot_v3_{source_suffix}_{TAXONOMY_SUFFIX}_n{PILOT_SAMPLE_SIZE}_seed42.jsonl"
if NEWS_PILOT.exists():
    print("Keeping existing pilot review file:", NEWS_PILOT)
else:
    pilot_rows = sample_news_records(silver_v2.iter_records(), sample_size=PILOT_SAMPLE_SIZE, seed=42)
    NEWS_PILOT.parent.mkdir(parents=True, exist_ok=True)
    with NEWS_PILOT.open("x", encoding="utf-8", newline="\n") as handle:
        for row in pilot_rows:
            item = {"id": row["id"], "text": str(row["text"]), "prompt": build_prompt(row, get_v3_domain_labels()), "pilot_only": True, "label": "", "needs_review": True, "approved": False, "reviewer": "", "reason": ""}
            handle.write(json.dumps(item, ensure_ascii=False) + "\n")
    print("Created pilot-only sample:", NEWS_PILOT, "| sampled rows:", len(pilot_rows))
print("Use this pilot to revise/freeze the taxonomy and guideline. It cannot publish Silver V3.")

## Optional local LLM suggestions (no API key)

Run this only after reviewing the pilot and finalizing the label guide. It downloads the public model weights, performs inference in this Colab GPU runtime, and writes unapproved suggestions to your mounted Drive; article text is not sent to an LLM API. The candidate file includes article text for review, so keep the Drive private and do not commit or share the file publicly. The source dataset license is marked unverified.

Enable a GPU in **Runtime → Change runtime type → GPU**. Inference resumes from checkpoints; `RESUME_LOCAL_MODEL_LABELING=True` is only for an unreviewed output from the same model/taxonomy. Long inputs are truncated to configured limits and explicitly marked for review.

In [ ]:
INSTALL_LOCAL_MODEL_DEPS = False  # Optional: install transformers/accelerate/bitsandbytes.
if INSTALL_LOCAL_MODEL_DEPS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[local-llm]"], cwd=PROJECT_DIR, check=True)

RUN_LOCAL_MODEL_LABELING = False  # Keep False until the pilot and label guide are approved.
RESUME_LOCAL_MODEL_LABELING = False
LOCAL_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
if RUN_LOCAL_MODEL_LABELING:
    command = [
        sys.executable, "scripts/15_label_news_v3_local.py",
        "--data-lake-root", str(DATA_LAKE_ROOT),
        "--silver-v2-report", SILVER_V2_REPORT_NAME,
        "--output", str(NEWS_DECISIONS),
        "--model-name", LOCAL_MODEL_NAME,
        "--batch-size", "2",
    ]
    if RESUME_LOCAL_MODEL_LABELING:
        command.append("--resume")
    subprocess.run(command, cwd=PROJECT_DIR, check=True)
else:
    print("Local LLM is optional and disabled. No API key is needed for this path.")

In [ ]:
CREATE_FULL_REVIEW_TEMPLATE = False  # Enable only after reviewing the pilot and freezing taxonomy.
if NEWS_DECISIONS.exists():
    print("Keeping existing review file; do not overwrite reviewed work:", NEWS_DECISIONS)
elif CREATE_FULL_REVIEW_TEMPLATE:
    with NEWS_DECISIONS.open("x", encoding="utf-8", newline="\n") as handle:
        for row in silver_v2.iter_records():
            if row["domain"] != "News":
                continue
            item = {"id": row["id"], "source_domain": "News", "text": row["text"], "prompt": build_prompt(row, get_v3_domain_labels()), "label": "", "needs_review": True, "approved": False, "reviewer": "", "reason": ""}
            handle.write(json.dumps(item, ensure_ascii=False) + "\n")
    print("Created full News review template:", NEWS_DECISIONS)
else:
    print("Pilot first. After finalizing the label guide, set CREATE_FULL_REVIEW_TEMPLATE=True and rerun this cell.")
print("Every full News row needs label/reviewer/reason, approved=true, needs_review=false.")
print("Candidate taxonomy:", get_v3_domain_labels())

In [ ]:
PUBLISH_SILVER_V3 = False  # Enable only after reviewing every full News decision.
silver_v3 = None
if not NEWS_DECISIONS.is_file():
    if PUBLISH_SILVER_V3:
        raise FileNotFoundError("Create the full News review file after finishing the pilot first")
    print("Silver V3 is not ready; create and review all News decisions first.")
else:
    silver_v3_pipeline = SilverV3Pipeline(silver_v2, NEWS_DECISIONS)
    decisions_sha256 = hashlib.sha256(NEWS_DECISIONS.read_bytes()).hexdigest()
    expected_silver = silver_v3_pipeline.planned_paths(verified_v2["sha256"], decisions_sha256)
    existing_silver = [path for path in expected_silver.values() if path.exists()]
    stale_silver = [path for path in expected_silver.values() if path.with_name(path.name + ".uploading").exists()]
    print("Silver V3 output:", {name: str(path) for name, path in expected_silver.items()})
    if stale_silver or (existing_silver and len(existing_silver) != len(expected_silver)):
        raise FileExistsError("Partial Silver V3 publication or stale upload; inspect Drive before retrying")
    if existing_silver:
        silver_v3 = SilverReader(DATA_LAKE_ROOT, expected_silver["report"].name)
        print("Existing Silver V3 verified:", SilverV3Pipeline.verify_reader(silver_v3))
    elif PUBLISH_SILVER_V3:
        silver_v3_pipeline.run()
        silver_v3 = SilverReader(DATA_LAKE_ROOT, expected_silver["report"].name)
        print("Silver V3 published and verified:", SilverV3Pipeline.verify_reader(silver_v3))
    else:
        print("No Silver V3 snapshot written. Review/edit decisions, then set PUBLISH_SILVER_V3=True.")

In [ ]:
PUBLISH_GOLD_V3 = False  # Enable only after Silver V3 has passed review and verification.
gold_v3 = None
if silver_v3 is None:
    print("Gold V3 is not ready: publish/select Silver V3 first.")
else:
    gold_pipeline = GoldV3Pipeline(silver_v3, config=GoldConfig(seed=42))
    verified_silver_v3 = SilverV3Pipeline.verify_reader(silver_v3)
    expected_gold = gold_pipeline.planned_paths(verified_silver_v3["sha256"])
    existing_gold = [path for path in expected_gold.values() if path.exists()]
    stale_gold = [path for path in expected_gold.values() if path.with_name(path.name + ".uploading").exists()]
    print("Gold V3 output:", {name: str(path) for name, path in expected_gold.items()})
    if stale_gold or (existing_gold and len(existing_gold) != len(expected_gold)):
        raise FileExistsError("Partial Gold V3 publication or stale upload; inspect Drive before retrying")
    if existing_gold:
        gold_v3 = GoldReader(DATA_LAKE_ROOT, expected_gold["manifest"].name)
        print("Existing Gold V3 verified:", gold_v3.verify())
    elif PUBLISH_GOLD_V3:
        result_gold = gold_pipeline.run()
        gold_v3 = GoldReader(DATA_LAKE_ROOT, expected_gold["manifest"].name)
        print("Gold V3 published:", result_gold["paths"])
        print("Gold V3 verified:", gold_v3.verify())
    else:
        print("No Gold V3 snapshot written. Review Silver V3 and set PUBLISH_GOLD_V3=True.")

In [ ]:
RUN_MODELING_V3 = False  # Enable only after the selected Gold V3 snapshot is verified.
if not RUN_MODELING_V3:
    print("Modeling is disabled. Publish and verify Gold V3 before enabling it.")
else:
    if gold_v3 is None:
        raise RuntimeError("Gold V3 is not available in this run; publish/select it first")
    from vietnamese_topic_classification.training import ClassifierConfig, ExperimentConfig, FeatureConfig
    from vietnamese_topic_classification.training.runner import run_experiment

    if gold_v3.version != "v3" or len(gold_v3.manifest["label_to_id"]) != 27:
        raise ValueError("Modeling requires a verified 27-class Gold V3 manifest")
    manifest_sha256 = hashlib.sha256(gold_v3.manifest_path.read_bytes()).hexdigest()
    source = gold_v3.manifest["source"]
    model_output_root = DATA_LAKE_ROOT / "experiments" / "tfidf_baselines_v3"
    model_specs = [
        ("tfidf_logreg", "logistic_regression"),
        ("tfidf_linear_svc", "linear_svc"),
    ]
    results = []
    for slug, classifier_kind in model_specs:
        experiment_name = f"{slug}_v3_{manifest_sha256[:10]}"
        artifact_path = model_output_root / experiment_name
        if artifact_path.exists():
            required = ("model.joblib", "config.json", "metadata.json", "validation_metrics.json")
            if any(not (artifact_path / name).is_file() for name in required):
                raise FileExistsError(f"Incomplete immutable experiment artifact: {artifact_path}")
            config_saved = json.loads((artifact_path / "config.json").read_text(encoding="utf-8"))
            metadata_saved = json.loads((artifact_path / "metadata.json").read_text(encoding="utf-8"))
            report_saved = json.loads((artifact_path / "validation_metrics.json").read_text(encoding="utf-8"))
            params_saved = config_saved.get("features", {}).get("params", {})
            confusion = report_saved.get("confusion_matrix", [])
            if (config_saved.get("name") != experiment_name
                    or config_saved.get("gold_manifest_name") != gold_v3.manifest_path.name
                    or config_saved.get("seed") != 42
                    or config_saved.get("selection_metric") != "macro_f1"
                    or config_saved.get("classifier", {}).get("kind") != classifier_kind
                    or config_saved.get("classifier", {}).get("params", {}).get("class_weight") is not None
                    or config_saved.get("features", {}).get("kind") != "tfidf"
                    or params_saved.get("ngram_range") != [1, 2]
                    or params_saved.get("min_df") != 2
                    or params_saved.get("max_features") != 100_000
                    or metadata_saved.get("experiment_name") != experiment_name
                    or metadata_saved.get("test_evaluated") is not False
                    or metadata_saved.get("lineage", {}).get("gold_manifest_name") != gold_v3.manifest_path.name
                    or metadata_saved.get("lineage", {}).get("gold_manifest_sha256") != manifest_sha256
                    or metadata_saved.get("lineage", {}).get("gold_source") != source
                    or (artifact_path / "model.joblib").stat().st_size == 0
                    or len(report_saved.get("per_class_f1", {})) != 27
                    or len(confusion) != 27
                    or any(len(row) != 27 for row in confusion)):
                raise ValueError(f"Existing experiment does not match verified Gold V3: {artifact_path}")
            metrics = {key: report_saved[key] for key in ("accuracy", "macro_f1", "weighted_f1")}
            print("Verified existing experiment:", experiment_name, metrics)
        else:
            result = run_experiment(
                ExperimentConfig(
                    name=experiment_name,
                    gold_manifest_name=gold_v3.manifest_path.name,
                    seed=42,
                    features=FeatureConfig(params={"ngram_range": (1, 2), "min_df": 2, "max_features": 100_000}),
                    classifier=ClassifierConfig(kind=classifier_kind, params={"class_weight": None}),
                ),
                data_lake_root=DATA_LAKE_ROOT,
                output_root=model_output_root,
            )
            metrics = result["validation_metrics"]
            print("Trained:", experiment_name, metrics)
        results.append((experiment_name, metrics))
    results.sort(key=lambda item: (-item[1]["macro_f1"], item[0]))
    print("Best VALIDATION Macro-F1:", results[0][0])
    print("Gold test is checksum-verified only; it is not loaded for training or validation.")

## Drive outputs

- `02_silver/v3/`: immutable Silver V3 JSONL.
- `04_reports/silver/v3/`: keyword report, pilot, full News review input, reviewed decisions, Silver V3 audit and quality report.
- `03_gold/v3/` and `04_reports/gold/v3/`: train/validation/test, manifest and quality report.

The full Silver V2 pool participates in taxonomy exploration before the Gold V3 split. Keep Gold test held out from model training/model selection, and report the full-pool taxonomy-development limitation. For shorter runs, use the separate Silver, Gold, and Modeling notebooks in this folder.